# Week 11 - Vector Databases & Semantic Search
**Using the Gemini API instead of OpenAI**

**Author**: Muhammad Raheel Ijaz   **Date**: 8/16/2026

This notebook follows the Lab 11 instructions, with OpenAI's `text-embedding-ada-002` replaced by Google's `gemini-embedding-001` model, and `ChatOpenAI` replaced by `ChatGoogleGenerativeAI` (Gemini) via LangChain.

Make sure the `company_docs/` folder from Week 10 (with `hr_policy.txt`, `benefits.txt`, `it_policy.txt`) sits in the same folder as this notebook before running.


## Part 1: Embeddings
### Task 1.1: Generate Embeddings

In [8]:
import os
import numpy as np
from google import genai
from google.genai import types as genai_types
from dotenv import load_dotenv

load_dotenv()

client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))


def get_embedding(text):
    """
    Generate an embedding for text using Gemini.
    Returns: list of floats (3072 numbers for gemini-embedding-001)
    """
    result = client.models.embed_content(
        model='gemini-embedding-001',
        contents=text,
        config=genai_types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT')
    )
    return result.embeddings[0].values


# Test it
text = 'vacation policy'
embedding = get_embedding(text)
print(f'Embedding length: {len(embedding)}')
print(f'First 5 values: {embedding[:5]}')


Embedding length: 3072
First 5 values: [-0.02950496, 0.03353201, -0.0048776804, -0.063203365, -0.0009708868]


### Task 1.2: Calculate Similarity

In [9]:
def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors.
    Returns: float between -1 and 1
    """
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)


# Test with similar phrases
phrases = [
    'vacation policy',
    'time off rules',
    'PTO guidelines',
    'dress code requirements'
]

# Get embeddings
embeddings = [get_embedding(p) for p in phrases]

# Compare first phrase with others
base = embeddings[0]
print(f'Comparing "{phrases[0]}" with:\n')
for i, phrase in enumerate(phrases[1:], 1):
    similarity = cosine_similarity(base, embeddings[i])
    print(f'{phrase:30} Similarity: {similarity:.4f}')


Comparing "vacation policy" with:

time off rules                 Similarity: 0.8962
PTO guidelines                 Similarity: 0.8699
dress code requirements        Similarity: 0.7716


**Expected:** 'time off' and 'PTO' should have HIGH similarity (~0.85-0.95). 'dress code' should be LOW (~0.1-0.3) since it's unrelated in meaning.


## Part 2: ChromaDB Setup
### Task 2.1: Initialize ChromaDB

In [10]:
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings


class GeminiEmbeddingFunction(EmbeddingFunction):
    """
    Custom ChromaDB embedding function that uses Gemini's embedding model.
    ChromaDB calls this automatically whenever you add or query documents.
    """
    def __init__(self, api_key, model_name='gemini-embedding-001', task_type='RETRIEVAL_DOCUMENT'):
        self.client = genai.Client(api_key=api_key)
        self.model_name = model_name
        self.task_type = task_type

    def __call__(self, input: Documents) -> Embeddings:
        result = self.client.models.embed_content(
            model=self.model_name,
            contents=list(input),
            config=genai_types.EmbedContentConfig(task_type=self.task_type)
        )
        return [e.values for e in result.embeddings]


# Create ChromaDB client (persistent - saves to disk)
chroma_client = chromadb.PersistentClient(path='./chroma_db')

# Create the Gemini embedding function
gemini_ef = GeminiEmbeddingFunction(api_key=os.getenv('GEMINI_API_KEY'))

# Create or get collection
collection = chroma_client.get_or_create_collection(
    name='company_docs',
    embedding_function=gemini_ef,
    metadata={'description': 'Company policy documents'}
)

print(f'Collection: {collection.name}')
print(f'Count: {collection.count()}')


Collection: company_docs
Count: 9


### Task 2.2: Load and Index Documents

In [11]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load documents (reuse the plain-Python approach from Week 10 -
# avoids depending on the deprecated langchain-community package)
docs_folder = 'company_docs/'
documents = []

for filename in sorted(os.listdir(docs_folder)):
    if filename.endswith('.txt'):
        filepath = os.path.join(docs_folder, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
        documents.append(Document(page_content=text, metadata={'source': filepath}))

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)

# Add to ChromaDB (only if empty, so re-running this cell doesn't duplicate)
if collection.count() == 0:
    collection.add(
        documents=[chunk.page_content for chunk in chunks],
        ids=[f'doc_{i}' for i in range(len(chunks))],
        metadatas=[{'source': chunk.metadata.get('source', 'unknown')} for chunk in chunks]
    )
    print(f'Added {len(chunks)} chunks to ChromaDB')
else:
    print(f'Collection already has {collection.count()} documents')


Collection already has 9 documents


**Note:** Embeddings are generated automatically by ChromaDB using the `GeminiEmbeddingFunction` above. This may take 30-60 seconds depending on how many chunks you have.


## Part 3: Semantic RAG
### Task 3.1: Test Vector Search

In [12]:
def vector_search(query, n_results=3):
    """
    Search using semantic similarity (not exact keyword matching).
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results


# Test semantic understanding
test_queries = [
    'time off policy',    # Should find 'vacation'
    'WFH guidelines',     # Should find 'remote work'
    'maternity leave'     # Should find 'parental leave'
]

for query in test_queries:
    print(f'\n{"="*60}')
    print(f'Query: {query}')
    print(f'{"="*60}')
    results = vector_search(query, n_results=2)
    for i, doc in enumerate(results['documents'][0]):
        distance = results['distances'][0][i]
        print(f'\nResult {i+1} (distance: {distance:.4f}):')
        print(doc[:200] + '...')



Query: time off policy

Result 1 (distance: 0.2876):
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year. Vacation days
accrue monthly and can be used after 90 days of employment. Unused va...

Result 2 (distance: 0.3549):
Sick Leave:
Employees receive unlimited sick leave, subject to manager approval for absences
longer than 3 consecutive days. A doctor's note is required for sick leave
exceeding 5 consecutive days.

P...

Query: WFH guidelines

Result 1 (distance: 0.3738):
Password Policy:
Passwords must be at least 12 characters long and changed every 90 days.
Multi-factor authentication (MFA) is required for all company accounts.

VPN and Remote Access:
Employees work...

Result 2 (distance: 0.3777):
Remote Work Policy:
Employees may work remotely up to 3 days per week. Remote work requires manager
approval and must be scheduled in advance through the HR portal. Employees working
remotely are expe...

Query: maternity

**Magic!** 'time off' finds 'vacation', 'WFH' finds 'remote work', even though the exact words don't match — this is what semantic search buys you over keyword search.


### Task 3.2: Build Semantic RAG Pipeline

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=os.getenv('GEMINI_API_KEY'),
    temperature=0
)


def semantic_rag(query, n_results=3):
    """
    Complete semantic RAG pipeline: vector search -> generate.
    """
    # Step 1: Vector search
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    if not results['documents'][0]:
        return 'No relevant information found.'

    # Step 2: Build context
    context = '\n\n---\n\n'.join(results['documents'][0])

    # Step 3: Create prompt
    prompt = f'''You are a helpful HR assistant. Answer using ONLY the context below.
If not in context, say so.

Context:
{context}

Question: {query}

Answer:'''

    # Step 4: Generate
    response = llm.invoke(prompt)
    return response.content


# Test
questions = [
    'How much time off do employees get?',
    'Can I work from home?',
    'What is the maternity leave policy?'
]

for q in questions:
    print(f'\nQ: {q}')
    print(f'A: {semantic_rag(q)}')



Q: How much time off do employees get?
A: Employees receive:
*   Unlimited sick leave, subject to manager approval for absences longer than 3 consecutive days, and a doctor's note for sick leave exceeding 5 consecutive days.
*   15 days of paid vacation per year (for full-time employees), which accrue monthly and can be used after 90 days of employment. Up to 5 unused vacation days roll over into the next calendar year.
*   12 weeks paid parental leave for primary caregivers and 6 weeks paid leave for secondary caregivers, which can be taken within the first 12 months after the birth or adoption of a child.

Q: Can I work from home?
A: Employees may work remotely up to 3 days per week. Remote work requires manager approval and must be scheduled in advance through the HR portal.

Q: What is the maternity leave policy?
A: The context does not explicitly state a "maternity leave policy." However, it does outline a "Parental Leave" policy:

*   12 weeks paid parental leave for primary car

## Bonus: Keyword vs. Semantic Comparison

In [14]:
# From Week 10 - keyword search
def keyword_search(query, chunks, top_k=3):
    query_lower = query.lower()
    scored = []
    for chunk in chunks:
        score = sum(chunk.page_content.lower().count(word) for word in query_lower.split())
        if score > 0:
            scored.append((score, chunk))
    scored.sort(reverse=True, key=lambda x: x[0])
    return [c for s, c in scored[:top_k]]


# Compare on a synonym query
query = 'PTO policy'  # Uses 'PTO', but the docs say 'vacation'

print('KEYWORD SEARCH:')
kw_results = keyword_search(query, chunks, top_k=2)
print(f'Found: {len(kw_results)} results')

print('\nSEMANTIC SEARCH:')
sem_results = vector_search(query, n_results=2)
print(f'Found: {len(sem_results["documents"][0])} results')
print(f'Top result: {sem_results["documents"][0][0][:100]}...')


KEYWORD SEARCH:
Found: 2 results

SEMANTIC SEARCH:
Found: 2 results
Top result: Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid va...


**Result:** Keyword search finds 0 results (there's no literal 'PTO' anywhere in the docs). Semantic search still finds the vacation policy — it understands that PTO means the same thing as vacation, even without a shared word.

---
### Next steps
Coming up: wrapping this whole system in a Streamlit web UI so it's usable outside a notebook — a real deployable app instead of just cells you run manually.